In [2]:
"""
PHASE 4.5: COMPLETE MODEL COMPARISON & ENHANCED EVALUATION

This notebook provides the COMPLETE comparison that's essential for your paper:
1. Pure CF (Collaborative Filtering Only)
2. Pure CB (Content-Based Only) - Word2Vec, BERT, Combined
3. Hybrid (CF + CB) - Word2Vec, BERT, Combined
4. Enhanced ranking metrics (Recall@K for multiple K values)
5. Comprehensive comparison tables and visualizations

This fills the gap from Phase 4 by adding pure content-based baselines!
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy import stats
import joblib
import warnings
import os
from collections import defaultdict
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context("paper", font_scale=1.5)
sns.set_palette("colorblind")

print("="*70)
print("PHASE 4.5: COMPLETE MODEL COMPARISON")
print("="*70)

# ============================================================
# PART 1: LOAD ALL DATA AND MODELS
# ============================================================
print("\n[1/10] Loading models and data...")

# Load all 3 similarity dictionaries
sim_dict_w2v = joblib.load('item_similarity_w2v_500k.pkl')
sim_dict_bert = joblib.load('item_similarity_bert_500k.pkl')
sim_dict_both = joblib.load('item_similarity_both_500k.pkl')
print("✓ Loaded 3 similarity dictionaries")

# Load CF components
user_factors = joblib.load('user_factors_500k.pkl')
item_factors = joblib.load('item_factors_500k.pkl')
user_to_idx = joblib.load('user_to_idx_500k.pkl')
item_to_idx = joblib.load('item_to_idx_500k.pkl')
user_bias = joblib.load('user_bias_500k.pkl')
item_bias = joblib.load('item_bias_500k.pkl')
global_stats = joblib.load('global_stats_500k.pkl')
global_mean = global_stats['global_mean']
print("✓ Loaded CF components")

# Load test set
df_test = pd.read_csv('test_data.csv')
df_test_filtered = df_test[
    (df_test['user_id'].isin(user_to_idx.keys())) & 
    (df_test['item_id'].isin(item_to_idx.keys()))
].copy()
print(f"✓ Test set loaded: {len(df_test_filtered):,} reviews")

# Load training data
df_train = pd.read_csv('train_data.csv')
df_train_sample = df_train.sample(n=500_000, random_state=42)

# Build lookups
user_items_lookup = df_train_sample.groupby('user_id')['item_id'].apply(set).to_dict()
user_item_ratings = {}
for uid in user_to_idx.keys():
    user_item_ratings[uid] = {}
for _, row in df_train_sample.iterrows():
    uid, iid = row['user_id'], row['item_id']
    if iid in item_to_idx:
        user_item_ratings[uid][item_to_idx[iid]] = row['rating']

print("✓ User lookups built")

# Sample for evaluation
test_sample = df_test_filtered.sample(n=min(20000, len(df_test_filtered)), random_state=42)
actual_test = test_sample['rating'].tolist()

# ============================================================
# PART 2: DEFINE PREDICTION FUNCTIONS
# ============================================================
print("\n[2/10] Defining prediction functions...")

def get_cf_score(user_id, item_id):
    """Pure Collaborative Filtering score"""
    if user_id not in user_to_idx or item_id not in item_to_idx:
        return global_mean
    
    u_idx = user_to_idx[user_id]
    i_idx = item_to_idx[item_id]
    
    dot = np.dot(user_factors[u_idx], item_factors[i_idx])
    pred = global_mean + user_bias.get(user_id, 0.0) + item_bias.get(item_id, 0.0) + dot
    
    return np.clip(pred, 1.0, 5.0)

def get_content_score(user_id, item_id, sim_dict):
    """Pure Content-Based score"""
    if user_id not in user_to_idx or item_id not in item_to_idx:
        return global_mean
    
    item_idx = item_to_idx[item_id]
    user_rated = user_items_lookup.get(user_id, set())
    
    if not user_rated or item_idx not in sim_dict:
        return global_mean
    
    user_item_indices = {item_to_idx[iid]: iid for iid in user_rated if iid in item_to_idx}
    if not user_item_indices:
        return global_mean
    
    similar_data = sim_dict[item_idx]
    weighted_sum, sim_sum = 0.0, 0.0
    
    for sim_idx, sim_val in zip(similar_data['indices'], similar_data['similarities']):
        if sim_idx in user_item_indices:
            rating = user_item_ratings[user_id].get(sim_idx, global_mean)
            weighted_sum += float(sim_val) * rating
            sim_sum += float(sim_val)
    
    return np.clip(weighted_sum / sim_sum, 1.0, 5.0) if sim_sum > 0 else global_mean

def get_hybrid_score(user_id, item_id, sim_dict, alpha=0.6):
    """Hybrid score combining CF and CB"""
    cf = get_cf_score(user_id, item_id)
    cb = get_content_score(user_id, item_id, sim_dict)
    return alpha * cf + (1 - alpha) * cb

print("✓ Prediction functions defined")

# ============================================================
# PART 3: EVALUATE ALL 6 APPROACHES
# ============================================================
print("\n[3/10] Evaluating ALL 6 approaches on test set...")
print("  This will take a few minutes...")

all_approaches = {}

# 1. CF Only
print("\n  [1/6] CF Only...")
cf_preds = [get_cf_score(row['user_id'], row['item_id']) for _, row in test_sample.iterrows()]
all_approaches['CF Only'] = {
    'predictions': cf_preds,
    'rmse': np.sqrt(mean_squared_error(actual_test, cf_preds)),
    'mae': mean_absolute_error(actual_test, cf_preds)
}

# 2-4. Pure Content-Based (CB Only)
for emb_name, sim_dict in [('Word2Vec', sim_dict_w2v), ('BERT', sim_dict_bert), ('Combined', sim_dict_both)]:
    print(f"  [CB Only - {emb_name}]...")
    cb_preds = [get_content_score(row['user_id'], row['item_id'], sim_dict) 
                for _, row in test_sample.iterrows()]
    all_approaches[f'CB Only ({emb_name})'] = {
        'predictions': cb_preds,
        'rmse': np.sqrt(mean_squared_error(actual_test, cb_preds)),
        'mae': mean_absolute_error(actual_test, cb_preds)
    }

# 5-7. Hybrid (CF + CB)
for emb_name, sim_dict in [('Word2Vec', sim_dict_w2v), ('BERT', sim_dict_bert), ('Combined', sim_dict_both)]:
    print(f"  [Hybrid - {emb_name}]...")
    hybrid_preds = [get_hybrid_score(row['user_id'], row['item_id'], sim_dict, alpha=0.6) 
                    for _, row in test_sample.iterrows()]
    all_approaches[f'Hybrid ({emb_name})'] = {
        'predictions': hybrid_preds,
        'rmse': np.sqrt(mean_squared_error(actual_test, hybrid_preds)),
        'mae': mean_absolute_error(actual_test, hybrid_preds)
    }

print("\n✓ All 7 approaches evaluated!")

# ============================================================
# PART 4: MAIN RESULTS TABLE
# ============================================================
print("\n[4/10] Generating main results table...")

print("\n" + "="*70)
print("COMPLETE MODEL COMPARISON RESULTS")
print("="*70)
print("\nApproach                | RMSE   | MAE    | vs CF  | vs Best CB")
print("------------------------|--------|--------|--------|------------")

cf_baseline = all_approaches['CF Only']['rmse']
best_cb_rmse = min(all_approaches[k]['rmse'] for k in all_approaches if 'CB Only' in k)

for approach_name in [
    'CF Only',
    'CB Only (Word2Vec)',
    'CB Only (BERT)',
    'CB Only (Combined)',
    'Hybrid (Word2Vec)',
    'Hybrid (BERT)',
    'Hybrid (Combined)'
]:
    metrics = all_approaches[approach_name]
    rmse = metrics['rmse']
    mae = metrics['mae']
    
    vs_cf = ((cf_baseline - rmse) / cf_baseline * 100)
    vs_cb = ((best_cb_rmse - rmse) / best_cb_rmse * 100) if 'CB Only' not in approach_name else 0
    
    vs_cf_str = f"{vs_cf:+.1f}%" if approach_name != 'CF Only' else "---"
    vs_cb_str = f"{vs_cb:+.1f}%" if 'CB Only' not in approach_name else "---"
    
    print(f"{approach_name:23} | {rmse:.4f} | {mae:.4f} | {vs_cf_str:6} | {vs_cb_str:11}")

# Find overall best
best_approach = min(all_approaches.items(), key=lambda x: x[1]['rmse'])
print(f"\n🏆 Best Overall: {best_approach[0]} (RMSE = {best_approach[1]['rmse']:.4f})")

# ============================================================
# PART 5: STATISTICAL SIGNIFICANCE TESTS
# ============================================================
print("\n[5/10] Statistical significance testing...")

print("\n" + "="*70)
print("STATISTICAL SIGNIFICANCE TESTS (Paired t-tests)")
print("="*70)

comparisons = [
    ('CF Only', 'CB Only (Combined)'),
    ('CF Only', 'Hybrid (Combined)'),
    ('CB Only (Combined)', 'Hybrid (Combined)'),
]

for method1, method2 in comparisons:
    errors1 = np.array(actual_test) - np.array(all_approaches[method1]['predictions'])
    errors2 = np.array(actual_test) - np.array(all_approaches[method2]['predictions'])
    
    sq_errors1 = errors1 ** 2
    sq_errors2 = errors2 ** 2
    
    t_stat, p_value = stats.ttest_rel(sq_errors1, sq_errors2)
    
    # Effect size
    mean_diff = np.mean(sq_errors1 - sq_errors2)
    pooled_std = np.sqrt((np.var(sq_errors1) + np.var(sq_errors2)) / 2)
    cohens_d = mean_diff / pooled_std
    
    significance = '***' if p_value < 0.001 else '**' if p_value < 0.01 else '*' if p_value < 0.05 else 'n.s.'
    
    print(f"\n{method1} vs {method2}:")
    print(f"  t-statistic: {t_stat:.4f}")
    print(f"  p-value: {p_value:.6f} {significance}")
    print(f"  Cohen's d: {cohens_d:.4f}")
    
    rmse1 = all_approaches[method1]['rmse']
    rmse2 = all_approaches[method2]['rmse']
    improvement = ((rmse1 - rmse2) / rmse1 * 100)
    print(f"  RMSE improvement: {improvement:+.2f}%")

# ============================================================
# PART 6: ENHANCED RANKING METRICS (Multiple K Values)
# ============================================================
print("\n[6/10] Computing ranking metrics for multiple K values...")

def precision_at_k(predictions, actuals, user_items_map, k=10, threshold=4.0):
    """Precision@K with proper user grouping"""
    precisions = []
    
    for user_id, items in user_items_map.items():
        if len(items) < k:
            continue
        
        items_sorted = sorted(items, key=lambda x: x[1], reverse=True)
        top_k = items_sorted[:k]
        
        relevant = sum(1 for _, _, actual in top_k if actual >= threshold)
        precisions.append(relevant / k)
    
    return np.mean(precisions) if precisions else 0.0

def recall_at_k(predictions, actuals, user_items_map, k=10, threshold=4.0):
    """Recall@K with proper user grouping"""
    recalls = []
    
    for user_id, items in user_items_map.items():
        if len(items) < k:
            continue
        
        total_relevant = sum(1 for _, _, actual in items if actual >= threshold)
        if total_relevant == 0:
            continue
        
        items_sorted = sorted(items, key=lambda x: x[1], reverse=True)
        top_k = items_sorted[:k]
        
        relevant_in_topk = sum(1 for _, _, actual in top_k if actual >= threshold)
        recalls.append(relevant_in_topk / total_relevant)
    
    return np.mean(recalls) if recalls else 0.0

def ndcg_at_k(predictions, actuals, user_items_map, k=10):
    """nDCG@K with proper user grouping"""
    ndcgs = []
    
    for user_id, items in user_items_map.items():
        if len(items) < k:
            continue
        
        items_sorted = sorted(items, key=lambda x: x[1], reverse=True)
        top_k = items_sorted[:k]
        
        dcg = sum((2**actual - 1) / np.log2(i + 2) for i, (_, _, actual) in enumerate(top_k))
        
        items_ideal = sorted(items, key=lambda x: x[2], reverse=True)[:k]
        idcg = sum((2**actual - 1) / np.log2(i + 2) for i, (_, _, actual) in enumerate(items_ideal))
        
        if idcg > 0:
            ndcgs.append(dcg / idcg)
    
    return np.mean(ndcgs) if ndcgs else 0.0

# Build user-item maps
user_items_maps = {}

for approach_name, metrics in all_approaches.items():
    user_items_map = defaultdict(list)
    
    for i, (_, row) in enumerate(test_sample.iterrows()):
        user_id = row['user_id']
        item_id = row['item_id']
        pred = metrics['predictions'][i]
        actual = actual_test[i]
        user_items_map[user_id].append((item_id, pred, actual))
    
    user_items_maps[approach_name] = user_items_map

# Compute for K = 5, 10, 20
k_values = [5, 10, 20]
ranking_results = defaultdict(dict)

print("\nComputing Precision, Recall, nDCG for K = 5, 10, 20...")

for approach_name in all_approaches.keys():
    print(f"  {approach_name}...")
    user_items_map = user_items_maps[approach_name]
    
    for k in k_values:
        p_k = precision_at_k(None, None, user_items_map, k=k, threshold=4.0)
        r_k = recall_at_k(None, None, user_items_map, k=k, threshold=4.0)
        n_k = ndcg_at_k(None, None, user_items_map, k=k)
        
        ranking_results[approach_name][f'P@{k}'] = p_k
        ranking_results[approach_name][f'R@{k}'] = r_k
        ranking_results[approach_name][f'nDCG@{k}'] = n_k

print("\n" + "="*70)
print("RANKING METRICS (Threshold = 4.0)")
print("="*70)

for k in k_values:
    print(f"\n📊 K = {k}")
    print(f"\nApproach                | P@{k}   | R@{k}   | nDCG@{k}")
    print("-" * 70)
    
    for approach_name in all_approaches.keys():
        p = ranking_results[approach_name][f'P@{k}']
        r = ranking_results[approach_name][f'R@{k}']
        n = ranking_results[approach_name][f'nDCG@{k}']
        print(f"{approach_name:23} | {p:.4f} | {r:.4f} | {n:.4f}")

# ============================================================
# PART 7: COLD-START ANALYSIS (ALL APPROACHES)
# ============================================================
print("\n[7/10] Cold-start item analysis...")

item_rating_counts = df_train_sample.groupby('item_id').size().to_dict()
test_sample['item_popularity'] = test_sample['item_id'].map(lambda x: item_rating_counts.get(x, 0))

cold_start_categories = [
    ('Cold (1-5)', 1, 5),
    ('Warm (6-20)', 6, 20),
    ('Popular (21+)', 21, 10000)
]

print("\n" + "="*70)
print("COLD-START ANALYSIS (ALL APPROACHES)")
print("="*70)

cold_results = {}

for cat_name, min_pop, max_pop in cold_start_categories:
    mask = (test_sample['item_popularity'] >= min_pop) & (test_sample['item_popularity'] <= max_pop)
    if mask.sum() == 0:
        continue
    
    indices = test_sample[mask].index
    seg_actual = [actual_test[i] for i, idx in enumerate(test_sample.index) if idx in indices]
    
    print(f"\n{cat_name} (n={len(seg_actual)}):")
    print(f"{'Approach':23} | RMSE")
    print("-" * 40)
    
    for approach_name, metrics in all_approaches.items():
        seg_preds = [metrics['predictions'][i] for i, idx in enumerate(test_sample.index) if idx in indices]
        rmse = np.sqrt(mean_squared_error(seg_actual, seg_preds))
        
        if cat_name not in cold_results:
            cold_results[cat_name] = {}
        cold_results[cat_name][approach_name] = rmse
        
        print(f"{approach_name:23} | {rmse:.4f}")

# ============================================================
# PART 8: SAVE COMPREHENSIVE RESULTS
# ============================================================
print("\n[8/10] Saving comprehensive results...")

comprehensive_results = {
    'all_approaches': {
        name: {
            'rmse': metrics['rmse'],
            'mae': metrics['mae']
        }
        for name, metrics in all_approaches.items()
    },
    'ranking_metrics': dict(ranking_results),
    'cold_start_analysis': cold_results,
    'test_sample_size': len(test_sample),
    'best_approach': best_approach[0]
}

joblib.dump(comprehensive_results, 'phase4_5_comprehensive_results.pkl')
print("✓ Results saved to 'phase4_5_comprehensive_results.pkl'")

# ============================================================
# PART 9: CREATE ENHANCED VISUALIZATIONS
# ============================================================
print("\n[9/10] Creating publication-quality figures...")

os.makedirs('figures_v4', exist_ok=True)

# Figure 1: Complete Comparison (All 7 Approaches)
fig, ax = plt.subplots(figsize=(14, 8))

approaches_ordered = [
    'CF Only',
    'CB Only (Word2Vec)',
    'CB Only (BERT)', 
    'CB Only (Combined)',
    'Hybrid (Word2Vec)',
    'Hybrid (BERT)',
    'Hybrid (Combined)'
]

rmses = [all_approaches[app]['rmse'] for app in approaches_ordered]
maes = [all_approaches[app]['mae'] for app in approaches_ordered]

x = np.arange(len(approaches_ordered))
width = 0.35

colors_rmse = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2']
colors_mae = ['#bcbd22', '#17becf', '#aec7e8', '#ffbb78', '#98df8a', '#ff9896', '#c5b0d5']

bars1 = ax.bar(x - width/2, rmses, width, label='RMSE', alpha=0.8, color=colors_rmse)
bars2 = ax.bar(x + width/2, maes, width, label='MAE', alpha=0.8, color=colors_mae)

ax.set_ylabel('Error', fontsize=14)
ax.set_title('Complete Model Comparison: CF vs CB vs Hybrid', fontsize=16, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([app.replace(' ', '\n') for app in approaches_ordered], fontsize=10)
ax.legend(fontsize=12)
ax.grid(axis='y', alpha=0.3)

for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('figures_v4/fig1_complete_comparison.png', dpi=300, bbox_inches='tight')
plt.close()
print("  ✓ Figure 1: Complete comparison")

# Figure 2: CF vs CB vs Hybrid (Grouped by Embedding)
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for idx, emb in enumerate(['Word2Vec', 'BERT', 'Combined']):
    ax = axes[idx]
    
    methods = ['CF Only', f'CB Only ({emb})', f'Hybrid ({emb})']
    rmses_emb = [all_approaches[m]['rmse'] for m in methods]
    
    bars = ax.bar(range(len(methods)), rmses_emb, alpha=0.8, 
                  color=['#1f77b4', '#ff7f0e', '#2ca02c'])
    
    ax.set_title(f'{emb} Embedding', fontsize=14, fontweight='bold')
    ax.set_ylabel('RMSE', fontsize=12)
    ax.set_xticks(range(len(methods)))
    ax.set_xticklabels(['CF', 'CB', 'Hybrid'], fontsize=11)
    ax.grid(axis='y', alpha=0.3)
    
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.suptitle('CF vs CB vs Hybrid (by Embedding Type)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('figures_v4/fig2_cf_cb_hybrid_comparison.png', dpi=300, bbox_inches='tight')
plt.close()
print("  ✓ Figure 2: CF vs CB vs Hybrid")

# Figure 3: Ranking Metrics Heatmap
fig, axes = plt.subplots(1, 3, figsize=(18, 8))

for idx, k in enumerate([5, 10, 20]):
    ax = axes[idx]
    
    # Build matrix
    metrics_matrix = []
    for approach in approaches_ordered:
        row = [
            ranking_results[approach][f'P@{k}'],
            ranking_results[approach][f'R@{k}'],
            ranking_results[approach][f'nDCG@{k}']
        ]
        metrics_matrix.append(row)
    
    metrics_matrix = np.array(metrics_matrix)
    
    im = ax.imshow(metrics_matrix, cmap='YlOrRd', aspect='auto', vmin=0, vmax=1)
    
    ax.set_xticks(range(3))
    ax.set_xticklabels([f'P@{k}', f'R@{k}', f'nDCG@{k}'], fontsize=11)
    ax.set_yticks(range(len(approaches_ordered)))
    ax.set_yticklabels([app.replace(' ', '\n') for app in approaches_ordered], fontsize=9)
    
    for i in range(len(approaches_ordered)):
        for j in range(3):
            text = ax.text(j, i, f'{metrics_matrix[i, j]:.3f}',
                          ha="center", va="center", color="black", fontsize=9)
    
    ax.set_title(f'K = {k}', fontsize=14, fontweight='bold')

plt.suptitle('Ranking Metrics Heatmap', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.colorbar(im, ax=axes, label='Score', fraction=0.02)
plt.savefig('figures_v4/fig3_ranking_heatmap.png', dpi=300, bbox_inches='tight')
plt.close()
print("  ✓ Figure 3: Ranking metrics heatmap")

# Figure 4: Cold-Start Performance
fig, ax = plt.subplots(figsize=(14, 8))

categories = list(cold_results.keys())
x = np.arange(len(categories))
width = 0.11

for idx, approach in enumerate(approaches_ordered):
    rmses_cold = [cold_results[cat].get(approach, 0) for cat in categories]
    offset = (idx - 3) * width
    ax.bar(x + offset, rmses_cold, width, label=approach, alpha=0.8)

ax.set_ylabel('RMSE', fontsize=14)
ax.set_title('Cold-Start Performance (All Approaches)', fontsize=16, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(categories)
ax.legend(fontsize=9, ncol=2)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('figures_v4/fig4_cold_start_all.png', dpi=300, bbox_inches='tight')
plt.close()
print("  ✓ Figure 4: Cold-start performance")

print("\n✓ All figures saved in 'figures_v4/' directory")

# ============================================================
# PART 10: FINAL SUMMARY FOR PAPER
# ============================================================
print("\n[10/10] Generating paper-ready summary...")

print("\n" + "="*70)
print("PAPER-READY SUMMARY - COMPLETE COMPARISON")
print("="*70)

print("\n📊 MAIN FINDINGS:")

print("\n1. MODEL PERFORMANCE HIERARCHY:")
cf_rmse = all_approaches['CF Only']['rmse']
best_cb = min((k, v['rmse']) for k, v in all_approaches.items() if 'CB Only' in k)
best_hybrid = min((k, v['rmse']) for k, v in all_approaches.items() if 'Hybrid' in k)

print(f"   CF Only:              {cf_rmse:.4f} RMSE")
print(f"   Best CB ({best_cb[0].split('(')[1][:-1]}):     {best_cb[1]:.4f} RMSE ({((cf_rmse - best_cb[1])/cf_rmse*100):+.1f}% vs CF)")
print(f"   Best Hybrid ({best_hybrid[0].split('(')[1][:-1]}): {best_hybrid[1]:.4f} RMSE ({((cf_rmse - best_hybrid[1])/cf_rmse*100):+.1f}% vs CF)")

print("\n2. KEY INSIGHTS:")
print(f"   ✓ Pure CB outperforms CF by {((cf_rmse - best_cb[1])/cf_rmse*100):.1f}%")
print(f"   ✓ Hybrid outperforms CF by {((cf_rmse - best_hybrid[1])/cf_rmse*100):.1f}%")
print(f"   ✓ Hybrid outperforms best CB by {((best_cb[1] - best_hybrid[1])/best_cb[1]*100):.1f}%")

print("\n3. EMBEDDING COMPARISON:")
for emb in ['Word2Vec', 'BERT', 'Combined']:
    cb_rmse = all_approaches[f'CB Only ({emb})']['rmse']
    hyb_rmse = all_approaches[f'Hybrid ({emb})']['rmse']
    print(f"   {emb:10} - CB: {cb_rmse:.4f}, Hybrid: {hyb_rmse:.4f}")

print("\n4. COLD-START PERFORMANCE:")
if 'Cold (1-5)' in cold_results:
    print(f"   For items with 1-5 ratings:")
    cf_cold = cold_results['Cold (1-5)']['CF Only']
    best_hybrid_cold = min(cold_results['Cold (1-5)'][k] for k in cold_results['Cold (1-5)'] if 'Hybrid' in k)
    print(f"   CF:          {cf_cold:.4f}")
    print(f"   Best Hybrid: {best_hybrid_cold:.4f} ({((cf_cold - best_hybrid_cold)/cf_cold*100):+.1f}% improvement)")

print("\n" + "="*70)
print("PHASE 4.5 COMPLETE!")

PHASE 4.5: COMPLETE MODEL COMPARISON

[1/10] Loading models and data...
✓ Loaded 3 similarity dictionaries
✓ Loaded CF components
✓ Test set loaded: 109,363 reviews
✓ User lookups built

[2/10] Defining prediction functions...
✓ Prediction functions defined

[3/10] Evaluating ALL 6 approaches on test set...
  This will take a few minutes...

  [1/6] CF Only...
  [CB Only - Word2Vec]...
  [CB Only - BERT]...
  [CB Only - Combined]...
  [Hybrid - Word2Vec]...
  [Hybrid - BERT]...
  [Hybrid - Combined]...

✓ All 7 approaches evaluated!

[4/10] Generating main results table...

COMPLETE MODEL COMPARISON RESULTS

Approach                | RMSE   | MAE    | vs CF  | vs Best CB
------------------------|--------|--------|--------|------------
CF Only                 | 0.9726 | 0.5968 | ---    | +14.3%     
CB Only (Word2Vec)      | 1.1346 | 0.8701 | -16.7% | ---        
CB Only (BERT)          | 1.1363 | 0.8725 | -16.8% | ---        
CB Only (Combined)      | 1.1366 | 0.8727 | -16.9% | ---    